In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
import shap
import lime
import lime.lime_tabular

c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:

gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [5]:
probs_benign = gbm.predict_proba(X_test)[:, 1]
preds = gbm.predict(X_test)

summary = pd.DataFrame({
    'true':     y_test.values,
    'pred':     preds,
    'p_benign': probs_benign,
}, index=X_test.index)

picks = []

# 1. Confidently malignant AND model got it right
conf_malig = summary[(summary['p_benign'] < 0.05) & (summary['true'] == 0)]
if len(conf_malig):
    picks.append(('confident malignant', conf_malig.index[0]))

# 2. Borderline — widen the band if the strict one is empty
for lo, hi in [(0.4, 0.6), (0.3, 0.7), (0.2, 0.8)]:
    border = summary[(summary['p_benign'] > lo) & (summary['p_benign'] < hi)]
    if len(border):
        picks.append((f'borderline ({lo}-{hi})', border.index[0]))
        break

# 3. Misclassified
miss = summary[summary['pred'] != summary['true']]
if len(miss):
    picks.append(('misclassified', miss.index[0]))

print("Selected patients:")
for label, idx in picks:
    r = summary.loc[idx]
    print(f"  [{label:>22}] idx={idx:<4d} "
          f"P(benign)={r['p_benign']:.3f}  "
          f"pred={'benign' if r['pred'] else 'malignant'}  "
          f"true={'benign' if r['true'] else 'malignant'}")

patients_to_explain = [idx for _, idx in picks]


Selected patients:
  [   confident malignant] idx=70   P(benign)=0.000  pred=malignant  true=malignant
  [  borderline (0.2-0.8)] idx=81   P(benign)=0.299  pred=malignant  true=benign
  [         misclassified] idx=81   P(benign)=0.299  pred=malignant  true=benign


In [6]:
# --- Method 3 (replaces CORELS): GOSDT sparse decision tree ---
from gosdt import GOSDTClassifier

# Binarize each continuous feature at its training median.
# Produces 30 indicator columns, e.g. "mean radius>=14.32".
medians = X_train.median()

def binarize(df, thresholds):
    out = pd.DataFrame(index=df.index)
    for col, thr in thresholds.items():
        out[f"{col}>={thr:.4f}"] = (df[col] >= thr).astype(int)
    return out

X_train_bin = binarize(X_train, medians)
X_test_bin  = binarize(X_test,  medians)

gosdt = GOSDTClassifier(
    regularization=0.02,   # bump up for a smaller tree, down for more leaves
    depth_budget=4,
    time_limit=60,
    verbose=False,
)
gosdt.fit(X_train_bin, y_train)

print(f"GOSDT train acc: {(gosdt.predict(X_train_bin) == y_train).mean():.3f}")
print(f"GOSDT test  acc: {(gosdt.predict(X_test_bin)  == y_test ).mean():.3f}")
print("\nLearned tree:")
print(gosdt)   # if blank in your version, try gosdt.trees_[0] or gosdt.tree_

GOSDT train acc: 0.921
GOSDT test  acc: 0.930

Learned tree:
GOSDTClassifier(depth_budget=4, regularization=0.02, time_limit=60)


c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [7]:
for idx in patients_to_explain:
    patient = X_test.loc[[idx]]
    pred = gbm.predict(patient)[0]
    prob = gbm.predict_proba(patient)[0]
    true_label = y_test[idx] if hasattr(y_test, '__getitem__') else y_test.iloc[idx]
    
    print(f"\n{'='*70}")
    print(f"Patient {idx}: Predicted={'benign' if pred else 'malignant'} "
          f"(P(benign)={prob[1]:.3f}), True={'benign' if true_label else 'malignant'}")
    print(f"{'='*70}")
    
    # --- Method 1: SHAP ---
    explainer = shap.TreeExplainer(gbm)
    shap_values = explainer.shap_values(patient)
    shap_importance = pd.Series(
        shap_values[0].flatten() if isinstance(shap_values, list) else shap_values.flatten(),
        index=data.feature_names
    ).abs().sort_values(ascending=False)
    print("\n[SHAP] Top 5 features by importance:")
    for feat in shap_importance.head(5).index:
        val = shap_values[0].flatten()[list(data.feature_names).index(feat)] if isinstance(shap_values, list) else shap_values.flatten()[list(data.feature_names).index(feat)]
        print(f"  {feat}: SHAP = {val:+.3f}")
    
    # --- Method 2: LIME ---
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train.values, feature_names=list(data.feature_names),
        class_names=list(data.target_names), mode='classification'
    )
    lime_exp = lime_explainer.explain_instance(
        patient.values.flatten(), gbm.predict_proba, num_features=5
    )
    print("\n[LIME] Top 5 feature explanations:")
    for feat, weight in lime_exp.as_list()[:5]:
        print(f"  {feat}: weight = {weight:+.3f}")
    
    # --- Method 3: Rule-based explanations (GOSDT + RIPPER) ---
    from wittgenstein import RIPPER
    rip = RIPPER(k=2)
    rip.fit(X_train, y_train, class_feat='target')
    rip_pred = rip.predict(patient)
    print(f"\n[RIPPER] Prediction: {'benign' if rip_pred[0] else 'malignant'}")
    print(f"  Applicable rules: {rip.ruleset_}")
    
    # --- GOSDT path for this patient ---
    patient_bin = X_test_bin.loc[[idx]]
    gosdt_pred = gosdt.predict(patient_bin)[0]
    print(f"\n[GOSDT] Prediction: {'benign' if gosdt_pred else 'malignant'}")

    # Show the binary conditions that are TRUE for this patient.
    # These are the building blocks the tree's path is drawn from.
    active = patient_bin.iloc[0]
    true_conds = active[active == 1].index.tolist()
    print(f"  Active conditions ({len(true_conds)} of {len(active)}):")
    for cond in true_conds[:8]:
        print(f"    {cond}")
    if len(true_conds) > 8:
        print(f"    ... and {len(true_conds) - 8} more")


Patient 70: Predicted=malignant (P(benign)=0.000), True=malignant

[SHAP] Top 5 features by importance:
  worst concave points: SHAP = -2.087
  worst perimeter: SHAP = -1.513
  mean concave points: SHAP = -1.501
  worst radius: SHAP = -1.377
  radius error: SHAP = -0.971

[LIME] Top 5 feature explanations:
  worst concave points > 0.16: weight = -0.422
  worst radius > 18.41: weight = -0.260
  worst perimeter > 124.65: weight = -0.218
  worst area > 1031.50: weight = -0.160
  radius error > 0.47: weight = -0.156


c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(



[RIPPER] Prediction: malignant
  Applicable rules: [[meanconcavepoints=0.011-0.018] V [meanconcavity=0.035-0.045^worstconcavity=0.14-0.18] V [meanconcavepoints=0.018-0.023] V [meanconcavepoints=<0.011] V [meanconcavepoints=0.033-0.047^worstconcavepoints=0.084-0.099] V [meanconcavepoints=0.028-0.033^meanradius=12.74-13.3] V [meanconcavepoints=0.023-0.028] V [meanconcavity=0.045-0.062] V [meanconcavepoints=0.033-0.047^concavityerror=0.038-0.046] V [worstconcavepoints=0.099-0.12^worstarea=686.6-784.04] V [meanconcavepoints=0.028-0.033^meancompactness=0.068-0.079] V [worsttexture=17.7-20.05^worstradius=16.04-17.35] V [meanarea=445.44-497.4^worstradius=13.32-13.98] V [worstradius=<11.19] V [meanconcavepoints=0.033-0.047^perimetererror=2.29-2.59] V [meantexture=<14.13^worstconcavepoints=0.12-0.15]]

[GOSDT] Prediction: malignant
  Active conditions (18 of 30):
    mean radius>=13.3000
    mean texture>=18.6800
    mean perimeter>=85.9800
    mean area>=551.7000
    mean compactness>=0.0910


c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(



[RIPPER] Prediction: benign
  Applicable rules: [[meanconcavepoints=0.011-0.018] V [meanconcavepoints=<0.011] V [meanconcavepoints=0.018-0.023] V [meanconcavepoints=0.023-0.028^worsttexture=<17.7] V [meanconcavepoints=0.028-0.033^worsttexture=20.05-21.91] V [meanconcavepoints=0.033-0.047^worstconcavepoints=0.084-0.099] V [meanconcavepoints=0.023-0.028^worstconcavity=0.094-0.14] V [meanconcavepoints=0.033-0.047^smoothnesserror=0.0069-0.0076] V [meanconcavepoints=0.028-0.033] V [meanconcavepoints=0.023-0.028^worstperimeter=86.45-91.27] V [meanconcavepoints=0.033-0.047] V [meanconcavepoints=0.023-0.028^worstradius=11.19-12.48] V [meantexture=<14.13^meanconcavepoints=0.062-0.085] V [meanconcavepoints=0.023-0.028^meanperimeter=85.98-91.28] V [worstconcavity=0.14-0.18] V [worstarea=599.3-686.6] V [meanradius=<10.26]]

[GOSDT] Prediction: malignant
  Active conditions (21 of 30):
    mean radius>=13.3000
    mean perimeter>=85.9800
    mean smoothness>=0.0946
    mean compactness>=0.0910
   

c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(



[RIPPER] Prediction: malignant
  Applicable rules: [[meanconcavepoints=0.011-0.018] V [meanconcavepoints=0.023-0.028] V [meanconcavepoints=0.018-0.023] V [meanconcavepoints=<0.011] V [meanconcavepoints=0.028-0.033^meanradius=12.74-13.3] V [meanconcavepoints=0.033-0.047] V [meanconcavepoints=0.028-0.033^smoothnesserror=0.0087-0.01] V [meanconcavepoints=0.028-0.033^meanarea=610.22-692.86] V [worstconcavepoints=0.099-0.12^meantexture=<14.13] V [worstconcavepoints=0.12-0.15^worsttexture=17.7-20.05] V [meanarea=445.44-497.4] V [worstconcavity=0.14-0.18] V [radiuserror=<0.18] V [worstarea=686.6-784.04^meantexture=18.68-19.82] V [radiuserror=0.28-0.32^fractaldimensionerror=0.0027-0.0032]]

[GOSDT] Prediction: malignant
  Active conditions (21 of 30):
    mean radius>=13.3000
    mean perimeter>=85.9800
    mean smoothness>=0.0946
    mean compactness>=0.0910
    mean concavity>=0.0615
    mean concave points>=0.0334
    mean symmetry>=0.1792
    mean fractal dimension>=0.0615
    ... and 13 

c:\Users\liamt\Documents\GitHub\NSAI\corels-env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
